# Recife Monitora — Preparação dos Dados

**Projeto 5 — Gestão de Tecnologia da Informação — CESAR School**

**A base utilizada neste projeto é sintética e não representa resultados reais das equipes, unidades ou profissionais da Secretaria de Saúde do Recife.**

## Objetivo

Esta etapa tem como objetivo tratar os problemas de qualidade identificados durante o Entendimento dos Dados, preparando os conjuntos para as etapas posteriores de análise e visualização.

Os tratamentos serão realizados de forma controlada e documentada, preservando os arquivos originais e respeitando as regras de negócio identificadas.

Entre os principais pontos a serem tratados estão:
- adequação dos tipos de dados;
- padronização de categorias;
- padronização dos ciclos;
- conversão de variáveis numéricas armazenadas como texto;
- tratamento de duplicidades;
- padronização de escalas;
- avaliação de valores ausentes e fora dos domínios esperados;
- revalidação das regras de negócio após o tratamento.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
CAMINHO_DADOS = Path("../data/raw")
equipes = pd.read_csv(CAMINHO_DADOS / "equipes.csv")
resultados = pd.read_csv(CAMINHO_DADOS / "resultados_ciclos.csv")
eixo3 = pd.read_csv(CAMINHO_DADOS / "eixo3_indicadores.csv")
nps = pd.read_csv(CAMINHO_DADOS / "nps_atributos.csv")
planos = pd.read_csv(CAMINHO_DADOS / "planos_acao.csv")

## 1. Preservação dos dados originais

Os conjuntos originais serão mantidos sem alterações durante esta etapa, para os procedimentos de preparação, serão criadas cópias independentes dos DataFrames. Essa estratégia permite comparar os dados antes e depois do tratamento e evita alterações acidentais nos dados brutos.

In [4]:
equipes_tratado = equipes.copy()
resultados_tratado = resultados.copy()
eixo3_tratado = eixo3.copy()
nps_tratado = nps.copy()
planos_tratado = planos.copy()

## 1. Preparação da tabela `equipes`

A tabela `equipes` contém os dados cadastrais das equipes e possui uma equipe por registro, identificada por `codigo_ine`.

Durante o Entendimento dos Dados foram identificados pontos que exigem preparação antes das análises:

- identificadores interpretados como valores numéricos;
- diferentes representações para `tipo_equipe`;
- diferentes representações para `distrito_sanitario`;
- valores ausentes em `populacao_adscrita`.

Os tratamentos a seguir buscam corrigir problemas de representação e tipos de dados sem alterar o significado das informações originais.

### 1.1 Adequação dos identificadores

- Os campos `codigo_ine` e `codigo_cnes` representam identificadores e não grandezas numéricas, apesar de terem sido interpretados automaticamente como números inteiros pelo Pandas, serão convertidos para texto.
- Essa conversão evita operações matemáticas inadequadas e preserva a natureza categórica dos identificadores.

In [5]:
equipes_tratado["codigo_ine"] = (
    equipes_tratado["codigo_ine"].astype(str)
)

equipes_tratado["codigo_cnes"] = (
    equipes_tratado["codigo_cnes"].astype(str)
)

resultados_tratado["codigo_ine"] = (
    resultados_tratado["codigo_ine"].astype(str)
)

eixo3_tratado["codigo_ine"] = (
    eixo3_tratado["codigo_ine"].astype(str)
)

nps_tratado["codigo_ine"] = (
    nps_tratado["codigo_ine"].astype(str)
)

planos_tratado["codigo_ine"] = (
    planos_tratado["codigo_ine"].astype(str)
)



### 1.2 Padronização do tipo de equipe

- Durante o entendimento dos dados foram identificadas 12 representações textuais para `tipo_equipe`, embora a variável possua apenas três categorias conceituais: `eSF`, `eSB` e `eMulti`.
- As diferenças eram decorrentes de espaços, uso de hífen e variações de maiúsculas e minúsculas, os valores foram normalizados e posteriormente mapeados para a nomenclatura padrão.

In [6]:
equipes_tratado["tipo_equipe"] = (
    equipes_tratado["tipo_equipe"]
    .str.strip()
    .str.replace("-", "", regex=False)
    .str.lower()
)

mapa_tipo_equipe = {
    "esf": "eSF",
    "esb": "eSB",
    "emulti": "eMulti"
}

equipes_tratado["tipo_equipe"] = (
    equipes_tratado["tipo_equipe"]
    .map(mapa_tipo_equipe)
)



In [7]:
print("Quantidade de categorias:",
      equipes_tratado["tipo_equipe"].nunique())

print("\nDistribuição após padronização:")
print(equipes_tratado["tipo_equipe"].value_counts())

print("\nValores ausentes após o tratamento:",
      equipes_tratado["tipo_equipe"].isna().sum())

Quantidade de categorias: 3

Distribuição após padronização:
tipo_equipe
eSF       89
eSB       27
eMulti    16
Name: count, dtype: int64

Valores ausentes após o tratamento: 0


### 1.3 Padronização do distrito sanitário

- Durante o entendimento dos dados, foram identificadas 19 representações textuais em `distrito_sanitario`, embora a estrutura da base considere apenas 8 Distritos Sanitários
- As diferenças de representação serão avaliadas e padronizadas para uma nomenclatura única, preservando a identificação correta dos Distritos Sanitários I a VIII.

In [8]:
print("Quantidade de representações:",
      equipes_tratado["distrito_sanitario"].nunique())

print("\nDistribuição antes da padronização:")
print(equipes_tratado["distrito_sanitario"].value_counts())

Quantidade de representações: 19

Distribuição antes da padronização:
distrito_sanitario
Distrito Sanitário VI      18
Distrito Sanitário V       15
Distrito Sanitário VIII    15
Distrito Sanitário VII     14
Distrito Sanitário I       13
Distrito Sanitário III     13
Distrito Sanitário IV      13
Distrito Sanitário II      12
ds v                        4
DISTRITO SANITÁRIO V        3
Distrito Sanitario IV       3
DISTRITO SANITÁRIO VI       2
ds ii                       1
DISTRITO SANITÁRIO I        1
DS III                      1
DISTRITO SANITÁRIO VII      1
DS VII                      1
Distrito Sanitario V        1
DS II                       1
Name: count, dtype: int64


In [9]:
mapa_distritos = {
    "Distrito Sanitário I": "Distrito Sanitário I",
    "DISTRITO SANITÁRIO I": "Distrito Sanitário I",

    "Distrito Sanitário II": "Distrito Sanitário II",
    "ds ii": "Distrito Sanitário II",
    "DS II": "Distrito Sanitário II",

    "Distrito Sanitário III": "Distrito Sanitário III",
    "DS III": "Distrito Sanitário III",

    "Distrito Sanitário IV": "Distrito Sanitário IV",
    "Distrito Sanitario IV": "Distrito Sanitário IV",

    "Distrito Sanitário V": "Distrito Sanitário V",
    "DISTRITO SANITÁRIO V": "Distrito Sanitário V",
    "Distrito Sanitario V": "Distrito Sanitário V",
    "ds v": "Distrito Sanitário V",

    "Distrito Sanitário VI": "Distrito Sanitário VI",
    "DISTRITO SANITÁRIO VI": "Distrito Sanitário VI",

    "Distrito Sanitário VII": "Distrito Sanitário VII",
    "DISTRITO SANITÁRIO VII": "Distrito Sanitário VII",
    "DS VII": "Distrito Sanitário VII",

    "Distrito Sanitário VIII": "Distrito Sanitário VIII"
}

equipes_tratado["distrito_sanitario"] = (
    equipes_tratado["distrito_sanitario"]
    .map(mapa_distritos)
)

In [10]:
print("Quantidade de distritos:",
      equipes_tratado["distrito_sanitario"].nunique())

print("\nDistribuição após padronização:")
print(equipes_tratado["distrito_sanitario"].value_counts())

print("\nValores ausentes após o tratamento:",
      equipes_tratado["distrito_sanitario"].isna().sum())

print("\nTotal de registros:",
      equipes_tratado["distrito_sanitario"].notna().sum())

Quantidade de distritos: 8

Distribuição após padronização:
distrito_sanitario
Distrito Sanitário V       23
Distrito Sanitário VI      20
Distrito Sanitário VII     16
Distrito Sanitário IV      16
Distrito Sanitário VIII    15
Distrito Sanitário I       14
Distrito Sanitário III     14
Distrito Sanitário II      14
Name: count, dtype: int64

Valores ausentes após o tratamento: 0

Total de registros: 132


### 1.4 Valores ausentes em `populacao_adscrita`

- Foram identificados 11 registros sem informação de população adscrita, correspondentes a 8,33% das equipes.
- Como não há informação suficiente para determinar os valores reais dessas populações, não será realizada imputação automática por média, mediana ou outro valor estimado nesta etapa.
- Os valores ausentes serão preservados, evitando a introdução de dados artificiais que possam influenciar análises posteriores.

In [11]:
equipes_tratado["populacao_adscrita"] = (
    equipes_tratado["populacao_adscrita"]
    .astype("Int64")
)

In [12]:
print("Tipo:",
      equipes_tratado["populacao_adscrita"].dtype)

print("Valores ausentes:",
      equipes_tratado["populacao_adscrita"].isna().sum())

print("Registros preenchidos:",
      equipes_tratado["populacao_adscrita"].notna().sum())

Tipo: Int64
Valores ausentes: 11
Registros preenchidos: 121


### 1.5 Validação final da tabela `equipes`

- Após os tratamentos realizados, a tabela será novamente validada para verificar a preservação da quantidade de registros, a unicidade da chave primária, a padronização das categorias e a manutenção controlada dos valores ausentes.

In [13]:
print("Quantidade de registros:", len(equipes_tratado))

print("INEs únicos:",
      equipes_tratado["codigo_ine"].nunique())

print("INEs duplicados:",
      equipes_tratado["codigo_ine"].duplicated().sum())

print("\nTipos de equipe:",
      equipes_tratado["tipo_equipe"].nunique())

print("Distritos sanitários:",
      equipes_tratado["distrito_sanitario"].nunique())

print("\nAusentes em população adscrita:",
      equipes_tratado["populacao_adscrita"].isna().sum())

print("\nValores ausentes por coluna:")
print(equipes_tratado.isna().sum())

Quantidade de registros: 132
INEs únicos: 132
INEs duplicados: 0

Tipos de equipe: 3
Distritos sanitários: 8

Ausentes em população adscrita: 11

Valores ausentes por coluna:
codigo_cnes            0
codigo_ine             0
nome_unidade           0
nome_equipe            0
distrito_sanitario     0
tipo_equipe            0
grupo                  0
populacao_adscrita    11
n_profissionais        0
dtype: int64


In [14]:
comparacao_equipes = pd.DataFrame({
    "Indicador": [
        "Representações de tipo_equipe",
        "Representações de distrito_sanitario",
        "INEs duplicados",
        "Ausentes em populacao_adscrita"
    ],
    "Antes": [
        equipes["tipo_equipe"].nunique(),
        equipes["distrito_sanitario"].nunique(),
        equipes["codigo_ine"].duplicated().sum(),
        equipes["populacao_adscrita"].isna().sum()
    ],
    "Depois": [
        equipes_tratado["tipo_equipe"].nunique(),
        equipes_tratado["distrito_sanitario"].nunique(),
        equipes_tratado["codigo_ine"].duplicated().sum(),
        equipes_tratado["populacao_adscrita"].isna().sum()
    ]
})

comparacao_equipes

,Indicador,Antes,Depois
0,Representações de tipo_equipe,12,3
1,Representações de distrito_sanitario,19,8
2,INEs duplicados,0,0
3,Ausentes em populacao_adscrita,11,11


## 3. Preparação da tabela `resultados_ciclos`

### 3.1 Padronização dos ciclos

- Durante o entendimento dos dados, foram identificadas 43 representações textuais diferentes na variável `ciclo`, embora a base contemple apenas 11 ciclos conceituais entre 2023.1 e 2026.2.
- Foram observadas diferentes formas de representar o mesmo ciclo, como `AAAA.Q`, `AAAA-Q`, `AAAAQ#` e `Q/AAAA`.
- Os valores serão padronizados para o formato `AAAA.Q`, permitindo a correta identificação das combinações equipe × ciclo e facilitando as validações entre tabelas.

In [15]:
print("Representações antes da padronização:",
      resultados_tratado["ciclo"].nunique())

print("\nValores encontrados:")
print(
    resultados_tratado["ciclo"]
    .value_counts()
    .sort_index()
)

Representações antes da padronização: 43

Valores encontrados:
ciclo
1/2023      4
1/2024      1
1/2025      2
1/2026      3
2/2023      4
2/2024      4
2/2025      5
2/2026      2
2023-1      6
2023-2      2
2023-3      3
2023.1     95
2023.2    104
2023.3    108
2023Q1      2
2023Q2      2
2023Q3      4
2024-1      1
2024-2      3
2024-3      1
2024.1    118
2024.2    118
2024.3    127
2024Q1      4
2024Q2      3
2024Q3      2
2025-1      4
2025-2      1
2025-3      6
2025.1    125
2025.2    124
2025.3    120
2025Q1      4
2025Q2      3
2025Q3      4
2026-1      3
2026-2      4
2026.1    127
2026.2    130
2026Q1      1
2026Q2      2
3/2024      1
3/2025      5
Name: count, dtype: int64


In [16]:
def padronizar_ciclo(ciclo):
    ciclo = str(ciclo).strip()

    if "/" in ciclo:
        quadrimestre, ano = ciclo.split("/")
        return f"{ano}.{quadrimestre}"

    ciclo = ciclo.replace("Q", ".")
    ciclo = ciclo.replace("-", ".")

    return ciclo

In [18]:
resultados_tratado["ciclo"] = (
    resultados_tratado["ciclo"]
    .apply(padronizar_ciclo)
)

In [19]:
print("Quantidade de ciclos após a padronização:",
      resultados_tratado["ciclo"].nunique())

print("\nCiclos encontrados:")
print(
    resultados_tratado["ciclo"]
    .value_counts()
    .sort_index()
)

print("\nValores ausentes:",
      resultados_tratado["ciclo"].isna().sum())

Quantidade de ciclos após a padronização: 11

Ciclos encontrados:
ciclo
2023.1    107
2023.2    112
2023.3    115
2024.1    124
2024.2    128
2024.3    131
2025.1    135
2025.2    133
2025.3    135
2026.1    134
2026.2    138
Name: count, dtype: int64

Valores ausentes: 0


In [20]:
ciclos_esperados = {
    "2023.1", "2023.2", "2023.3",
    "2024.1", "2024.2", "2024.3",
    "2025.1", "2025.2", "2025.3",
    "2026.1", "2026.2"
}

ciclos_encontrados = set(
    resultados_tratado["ciclo"].dropna().unique()
)

print("Ciclos não esperados:",
      ciclos_encontrados - ciclos_esperados)

print("Ciclos esperados ausentes:",
      ciclos_esperados - ciclos_encontrados)

Ciclos não esperados: set()
Ciclos esperados ausentes: set()


### 2.2 Conversão de `eixo2_total`

- A variável `eixo2_total` representa uma pontuação numérica com limite máximo de 200 pontos, porém foi interpretada como texto devido à presença de diferentes separadores decimais.
- Foram identificados registros utilizando vírgula como separador decimal. Para permitir cálculos e validações posteriores, os valores serão padronizados e convertidos para tipo numérico.
- Os valores ausentes serão preservados, pois sua ausência possui significado distinto de uma pontuação igual a zero.

In [21]:
print("Tipo antes do tratamento:",
      resultados_tratado["eixo2_total"].dtype)

print("Valores ausentes antes:",
      resultados_tratado["eixo2_total"].isna().sum())

print("Valores com vírgula:",
      resultados_tratado["eixo2_total"]
      .str.contains(",", na=False)
      .sum())

Tipo antes do tratamento: str
Valores ausentes antes: 230
Valores com vírgula: 125


In [22]:
resultados_tratado["eixo2_total"] = pd.to_numeric(
    resultados_tratado["eixo2_total"]
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

In [23]:
print("Tipo depois do tratamento:",
      resultados_tratado["eixo2_total"].dtype)

print("Valores ausentes depois:",
      resultados_tratado["eixo2_total"].isna().sum())

print("Mínimo:",
      resultados_tratado["eixo2_total"].min())

print("Máximo:",
      resultados_tratado["eixo2_total"].max())

print("Valores fora de 0–200:",
      (~resultados_tratado["eixo2_total"]
       .between(0, 200)
       & resultados_tratado["eixo2_total"].notna())
      .sum())

Tipo depois do tratamento: float64
Valores ausentes depois: 230
Mínimo: 49.9
Máximo: 198.9
Valores fora de 0–200: 0


### 2.3 Avaliação das duplicidades

- A chave esperada da tabela `resultados_ciclos` é composta por `codigo_ine + ciclo`, pois cada equipe deve possuir no máximo um resultado por ciclo.
- Como os ciclos foram padronizados anteriormente, a existência de duplicidades será reavaliada após o tratamento. Essa verificação é necessária porque diferentes representações textuais do mesmo ciclo poderiam ocultar registros referentes à mesma combinação equipe × ciclo.
- As duplicidades serão analisadas antes de qualquer exclusão, evitando a remoção automática de registros potencialmente divergentes.

In [24]:
print(
    "Linhas duplicadas pela chave:",
    resultados_tratado.duplicated(
        subset=["codigo_ine", "ciclo"],
        keep=False
    ).sum()
)

print(
    "Duplicidades exatas:",
    resultados_tratado.duplicated().sum()
)

Linhas duplicadas pela chave: 32
Duplicidades exatas: 14


In [25]:
duplicados_chave = resultados_tratado[
    resultados_tratado.duplicated(
        subset=["codigo_ine", "ciclo"],
        keep=False
    )
].sort_values(["codigo_ine", "ciclo"])

duplicados_chave

,codigo_ine,ciclo,eixo1_bloco_a,eixo1_bloco_b,eixo1_bloco_c,eixo1_total,taxa_resposta_bloco_a,bloco_a_valido,eixo2_total,eixo3_total,pontuacao_final,zona_classificacao,situacao_participacao
13,6000000001,2023.3,50.4,39.7,28.8,118.9,0.6042,True,NaN,302.0,565.3,Aperfeiçoamento,Participante
1372,6000000001,2023.3,50.4,39.7,28.8,118.9,0.6042,True,NaN,302.0,565.3,Aperfeiçoamento,Participante
17,6000000001,2025.1,46.4,33.9,32.1,112.4,0.5851,True,120.6,393.5,626.5,Aperfeiçoamento,Participante
1366,6000000001,2025.1,46.4,33.9,32.1,112.4,0.5851,True,120.6,393.5,626.5,Aperfeiçoamento,Participante
326,6000000031,2024.2,0.0,45.7,34.9,80.6,0.2193,False,130.0,385.2,595.8,Aperfeiçoamento,Participante
1368,6000000031,2024.2,0.0,45.7,34.9,80.6,0.2193,False,130.0,385.2,595.8,Aperfeiçoamento,Participante
517,6000000048,2025.3,31.4,27.0,28.0,86.4,0.5195,True,88.8,302.5,477.7,Crítica,Participante
1375,6000000048,2025.3,31.4,27.0,28.0,86.4,0.5195,True,88.8,302.5,477.7,Crítica,Participante
586,6000000055,2025.3,49.3,38.4,45.2,132.9,0.3866,True,101.9,367.7,602.5,Aperfeiçoamento,Participante
1373,6000000055,2025.3,49.3,38.4,45.2,132.9,0.3866,True,101.9,367.7,602.5,Aperfeiçoamento,Participante


In [26]:
colunas_comparacao = [
    coluna for coluna in resultados_tratado.columns
    if coluna not in ["codigo_ine", "ciclo"]
]

analise_duplicados = (
    duplicados_chave
    .groupby(["codigo_ine", "ciclo"])[colunas_comparacao]
    .nunique(dropna=False)
)

pares_com_divergencia = (
    analise_duplicados > 1
).any(axis=1)

print("Total de pares duplicados:",
      len(analise_duplicados))

print("Pares sem divergência:",
      (~pares_com_divergencia).sum())

print("Pares com divergência:",
      pares_com_divergencia.sum())

Total de pares duplicados: 16
Pares sem divergência: 14
Pares com divergência: 2


In [27]:
divergencias_por_coluna = (
    analise_duplicados.loc[pares_com_divergencia] > 1
)

divergencias_por_coluna

,,eixo1_bloco_a,eixo1_bloco_b,eixo1_bloco_c,eixo1_total,taxa_resposta_bloco_a,bloco_a_valido,eixo2_total,eixo3_total,pontuacao_final,zona_classificacao,situacao_participacao
codigo_ine,ciclo,,,,,,,,,,,
6000000057,2023.3,False,False,False,False,False,False,True,False,False,False,False
6000000069,2023.2,False,False,False,False,False,False,True,False,False,False,False


#### Tratamento das duplicidades

- Após a padronização dos ciclos, foram identificadas 16 combinações duplicadas de `codigo_ine + ciclo`.Dessas, 14 apresentavam registros sem divergências nos demais atributos. Em 2 combinações, a única divergência identificada estava em `eixo2_total`, sendo que um registro apresentava valor ausente e o outro possuía a pontuação preenchida.
- Como critério de tratamento, foi priorizado o registro mais completo dentro de cada combinação equipe × ciclo. Dessa forma, nos casos divergentes foi preservado o registro com `eixo2_total` preenchido. Nos demais casos, uma das cópias redundantes foi mantida.
- Essa estratégia evita a perda de informação disponível e garante a unicidade da chave `codigo_ine + ciclo`.

In [28]:
resultados_tratado = (
    resultados_tratado
    .sort_values(
        by="eixo2_total",
        na_position="last"
    )
    .drop_duplicates(
        subset=["codigo_ine", "ciclo"],
        keep="first"
    )
    .reset_index(drop=True)
)

In [29]:
print("Quantidade de registros após o tratamento:",
      len(resultados_tratado))

print(
    "Chaves duplicadas após o tratamento:",
    resultados_tratado.duplicated(
        subset=["codigo_ine", "ciclo"]
    ).sum()
)

print(
    "Combinações únicas equipe × ciclo:",
    resultados_tratado[
        ["codigo_ine", "ciclo"]
    ].drop_duplicates().shape[0]
)

Quantidade de registros após o tratamento: 1376
Chaves duplicadas após o tratamento: 0
Combinações únicas equipe × ciclo: 1376


In [30]:
resultados_tratado[
    (
        (resultados_tratado["codigo_ine"] == "6000000057") &
        (resultados_tratado["ciclo"] == "2023.3")
    )
    |
    (
        (resultados_tratado["codigo_ine"] == "6000000069") &
        (resultados_tratado["ciclo"] == "2023.2")
    )
][["codigo_ine", "ciclo", "eixo2_total"]]

,codigo_ine,ciclo,eixo2_total
435,6000000057,2023.3,122.4
1083,6000000069,2023.2,170.6


### 2.4 Validação da taxa de resposta do Bloco A

- A variável `taxa_resposta_bloco_a` representa uma proporção e, portanto, deve apresentar valores entre 0 e 1.
- Além disso, conforme a regra de negócio, o Bloco A é considerado válido quando a taxa de resposta é igual ou superior a 30% (`0.30`).
- Durante o entendimento dos dados foram identificados valores fora do domínio esperado. Nesta etapa, esses registros serão avaliados antes da definição do tratamento, evitando correções arbitrárias.

In [32]:
taxa_fora_dominio = (
    ~resultados_tratado["taxa_resposta_bloco_a"].between(0, 1)
    & resultados_tratado["taxa_resposta_bloco_a"].notna()
)

print("Registros com taxa fora de 0–1:",
      taxa_fora_dominio.sum())

resultados_tratado.loc[
    taxa_fora_dominio,
    [
        "codigo_ine",
        "ciclo",
        "taxa_resposta_bloco_a",
        "bloco_a_valido",
        "eixo1_bloco_a",
        "eixo1_total",
        "situacao_participacao"
    ]
]

Registros com taxa fora de 0–1: 7


,codigo_ine,ciclo,taxa_resposta_bloco_a,bloco_a_valido,eixo1_bloco_a,eixo1_total,situacao_participacao
292,6000000028,2026.1,-0.10,True,56.3,136.0,Participante
608,6000000066,2023.2,1.35,True,35.7,92.9,Participante
670,6000000127,2023.2,-0.10,True,43.8,130.5,Participante
703,6000000123,2025.3,-0.10,True,60.8,147.3,Participante
788,6000000101,2025.1,2.40,True,66.8,145.1,Participante
909,6000000109,2023.2,-0.10,True,58.2,148.2,Participante
1114,6000000025,2024.3,2.40,True,50.3,131.1,Participante


In [33]:
taxa_dominio_valido = (
    resultados_tratado["taxa_resposta_bloco_a"].between(0, 1)
)

bloco_a_esperado = (
    resultados_tratado["taxa_resposta_bloco_a"] >= 0.30
)

inconsistencia_bloco_a = (
    taxa_dominio_valido
    & (resultados_tratado["bloco_a_valido"] != bloco_a_esperado)
)

print(
    "Inconsistências da regra dos 30% entre taxas válidas:",
    inconsistencia_bloco_a.sum()
)

Inconsistências da regra dos 30% entre taxas válidas: 0


#### Tratamento das taxas fora do domínio

- Foram identificados 7 registros com `taxa_resposta_bloco_a` fora do intervalo válido de 0 a 1.
- Como não existem informações suficientes para determinar o valor correto dessas taxas, não foi realizada imputação ou correção por aproximação. Os valores inválidos foram substituídos por ausentes (`NaN`), preservando as demais informações dos registros.
- A variável `bloco_a_valido` não foi recalculada para esses casos, pois não é possível determinar sua validade exclusivamente a partir de uma taxa inválida. Entre os registros com taxas dentro do domínio esperado, não foram encontradas inconsistências na aplicação da regra mínima de 30%.

In [34]:
resultados_tratado.loc[
    taxa_fora_dominio,
    "taxa_resposta_bloco_a"
] = np.nan

In [35]:
print(
    "Taxas fora de 0–1 após o tratamento:",
    (
        ~resultados_tratado["taxa_resposta_bloco_a"].between(0, 1)
        & resultados_tratado["taxa_resposta_bloco_a"].notna()
    ).sum()
)

print(
    "Valores ausentes em taxa_resposta_bloco_a:",
    resultados_tratado["taxa_resposta_bloco_a"].isna().sum()
)

Taxas fora de 0–1 após o tratamento: 0
Valores ausentes em taxa_resposta_bloco_a: 7


### 2.5 Pontuação do Bloco A em registros não válidos

- Conforme a regra de negócio, o Bloco A é considerado válido quando a taxa de resposta é igual ou superior a 30%, quando esse limite não é atingido, a pontuação do bloco não deve ser computada.
- Durante o entendimento dos dados foram observadas diferentes representações para registros com `bloco_a_valido = False`, incluindo pontuação igual a zero e valores ausentes.
- Antes de realizar qualquer padronização, será analisada a forma como esses casos estão representados na base após os tratamentos anteriores.

In [36]:
bloco_a_nao_valido = (
    resultados_tratado["bloco_a_valido"] == False
)

print(
    "Registros com Bloco A não válido:",
    bloco_a_nao_valido.sum()
)

print("\nRepresentação da pontuação:")
print(
    resultados_tratado.loc[
        bloco_a_nao_valido,
        "eixo1_bloco_a"
    ]
    .value_counts(dropna=False)
)

Registros com Bloco A não válido: 154

Representação da pontuação:
eixo1_bloco_a
0.0    148
NaN      6
Name: count, dtype: int64


In [37]:
print(
    "Pontuação igual a zero:",
    (
        resultados_tratado.loc[
            bloco_a_nao_valido,
            "eixo1_bloco_a"
        ] == 0
    ).sum()
)

print(
    "Pontuação ausente:",
    resultados_tratado.loc[
        bloco_a_nao_valido,
        "eixo1_bloco_a"
    ].isna().sum()
)

print(
    "Pontuação diferente de zero e não ausente:",
    (
        resultados_tratado.loc[
            bloco_a_nao_valido,
            "eixo1_bloco_a"
        ].notna()
        &
        (
            resultados_tratado.loc[
                bloco_a_nao_valido,
                "eixo1_bloco_a"
            ] != 0
        )
    ).sum()
)

Pontuação igual a zero: 148
Pontuação ausente: 6
Pontuação diferente de zero e não ausente: 0


#### Decisão de tratamento

- Foram identificados 154 registros com `bloco_a_valido = False`. Desses, 148 apresentam `eixo1_bloco_a = 0` e 6 apresentam valor ausente. Não foram encontrados casos de Bloco A não válido com pontuação positiva.
- Embora existam duas formas de representar a não computação do Bloco A, os valores foram preservados nesta etapa. A conversão dos zeros para ausentes poderia interferir em cálculos posteriores do `eixo1_total`, enquanto a substituição dos valores ausentes por zero caracterizaria uma imputação sem evidência suficiente.
- A inconsistência de representação foi documentada, sem alteração dos valores originais.

### 2.6 Validação das pontuações

As pontuações dos eixos e a pontuação final possuem limites definidos pelas regras do programa:

- Eixo 1: 0 a 200 pontos;
- Eixo 2: 0 a 200 pontos;
- Eixo 3: 0 a 600 pontos;
- Pontuação final: 0 a 1.000 pontos.

Após os tratamentos anteriores, os valores serão novamente avaliados quanto ao domínio esperado antes da definição de qualquer correção.

In [38]:
limites_pontuacao = {
    "eixo1_total": (0, 200),
    "eixo2_total": (0, 200),
    "eixo3_total": (0, 600),
    "pontuacao_final": (0, 1000)
}

for coluna, (minimo, maximo) in limites_pontuacao.items():

    fora_dominio = (
        resultados_tratado[coluna].notna()
        & ~resultados_tratado[coluna].between(minimo, maximo)
    )

    print(
        f"{coluna}: {fora_dominio.sum()} valores fora de "
        f"{minimo}–{maximo}"
    )

eixo1_total: 0 valores fora de 0–200
eixo2_total: 0 valores fora de 0–200
eixo3_total: 0 valores fora de 0–600
pontuacao_final: 9 valores fora de 0–1000


### 2.7 Consistência entre os eixos e a pontuação final

- Além da validação dos limites individuais, a pontuação final deve corresponder à soma das pontuações dos três eixos.
- Nesta etapa, será comparada a `pontuacao_final` registrada com a soma de `eixo1_total`, `eixo2_total` e `eixo3_total`.
- A comparação será realizada apenas nos registros em que os três eixos e a pontuação final estejam disponíveis.

In [39]:
resultados_tratado["pontuacao_calculada"] = (
    resultados_tratado["eixo1_total"]
    + resultados_tratado["eixo2_total"]
    + resultados_tratado["eixo3_total"]
)

resultados_tratado["diferenca_pontuacao"] = (
    resultados_tratado["pontuacao_final"]
    - resultados_tratado["pontuacao_calculada"]
).abs()

In [40]:
inconsistencia_pontuacao = (
    resultados_tratado["pontuacao_calculada"].notna()
    & resultados_tratado["pontuacao_final"].notna()
    & (resultados_tratado["diferenca_pontuacao"] > 0.01)
)

print(
    "Registros com pontuação final divergente da soma dos eixos:",
    inconsistencia_pontuacao.sum()
)

Registros com pontuação final divergente da soma dos eixos: 7


In [41]:
resultados_tratado.loc[
    inconsistencia_pontuacao,
    [
        "codigo_ine",
        "ciclo",
        "eixo1_total",
        "eixo2_total",
        "eixo3_total",
        "pontuacao_final",
        "pontuacao_calculada",
        "diferenca_pontuacao"
    ]
].sort_values("diferenca_pontuacao", ascending=False)

,codigo_ine,ciclo,eixo1_total,eixo2_total,eixo3_total,pontuacao_final,pontuacao_calculada,diferenca_pontuacao
301,6000000047,2024.2,128.9,114.3,389.2,9999.0,632.4,9366.6
346,6000000117,2026.2,116.2,117.4,413.8,9999.0,647.4,9351.6
247,6000000008,2025.3,86.3,110.2,317.6,1580.0,514.1,1065.9
474,6000000024,2023.3,86.0,124.5,349.5,1580.0,560.0,1020.0
490,6000000117,2025.1,119.9,125.5,365.4,1247.0,610.8,636.2
1139,6000000002,2025.2,89.5,191.5,465.4,1247.0,746.4,500.6
6,6000000098,2023.2,77.7,63.3,219.2,-35.0,360.2,395.2


In [42]:
pontuacao_fora_dominio = (
    resultados_tratado["pontuacao_final"].notna()
    & ~resultados_tratado["pontuacao_final"].between(0, 1000)
)

resultados_tratado.loc[
    pontuacao_fora_dominio,
    [
        "codigo_ine",
        "ciclo",
        "eixo1_total",
        "eixo2_total",
        "eixo3_total",
        "pontuacao_final",
        "pontuacao_calculada"
    ]
]

,codigo_ine,ciclo,eixo1_total,eixo2_total,eixo3_total,pontuacao_final,pontuacao_calculada
6,6000000098,2023.2,77.7,63.3,219.2,-35.0,360.2
247,6000000008,2025.3,86.3,110.2,317.6,1580.0,514.1
301,6000000047,2024.2,128.9,114.3,389.2,9999.0,632.4
346,6000000117,2026.2,116.2,117.4,413.8,9999.0,647.4
474,6000000024,2023.3,86.0,124.5,349.5,1580.0,560.0
490,6000000117,2025.1,119.9,125.5,365.4,1247.0,610.8
1139,6000000002,2025.2,89.5,191.5,465.4,1247.0,746.4
1201,6000000028,2023.3,97.9,NaN,323.5,-35.0,NaN
1335,6000000110,2023.3,123.5,NaN,367.5,-35.0,NaN


#### Tratamento das inconsistências da pontuação final

- Foram identificados 9 registros com `pontuacao_final` fora do intervalo permitido de 0 a 1.000 pontos.
- Em 7 desses registros, as pontuações dos três eixos estavam disponíveis, como a pontuação final corresponde à soma dos Eixos 1, 2 e 3, esses valores puderam ser corrigidos de forma determinística a partir dos componentes disponíveis.
- Nos 2 registros restantes, o `eixo2_total` estava ausente, impossibilitando o recálculo da pontuação final. Como os valores registrados de `-35` são inválidos e não existem informações suficientes para determinar a pontuação correta, eles foram substituídos por `NaN`.
- Dessa forma, valores foram corrigidos apenas quando havia evidência suficiente para determinar o resultado correto, evitando imputações arbitrárias.

In [43]:
resultados_tratado.loc[
    inconsistencia_pontuacao,
    "pontuacao_final"
] = resultados_tratado.loc[
    inconsistencia_pontuacao,
    "pontuacao_calculada"
]

In [44]:
pontuacao_ainda_invalida = (
    resultados_tratado["pontuacao_final"].notna()
    & ~resultados_tratado["pontuacao_final"].between(0, 1000)
)

print(
    "Pontuações ainda fora do domínio:",
    pontuacao_ainda_invalida.sum()
)

Pontuações ainda fora do domínio: 2


In [45]:
resultados_tratado.loc[
    pontuacao_ainda_invalida,
    "pontuacao_final"
] = np.nan

In [46]:
print(
    "Pontuações fora de 0–1000 após tratamento:",
    (
        resultados_tratado["pontuacao_final"].notna()
        & ~resultados_tratado["pontuacao_final"].between(0, 1000)
    ).sum()
)

print(
    "Valores ausentes em pontuacao_final:",
    resultados_tratado["pontuacao_final"].isna().sum()
)

Pontuações fora de 0–1000 após tratamento: 0
Valores ausentes em pontuacao_final: 60


### 2.8 Validação da zona de classificação

- A `zona_classificacao` é determinada diretamente pela `pontuacao_final`, conforme as faixas definidas pelas regras de negócio.
- Como alguns valores de `pontuacao_final` foram corrigidos na etapa anterior, as classificações serão novamente validadas para garantir consistência entre a pontuação e a respectiva zona.
- Registros sem pontuação final não terão sua classificação inferida.
- Foram identificados 2 registros sem `pontuacao_final` que ainda possuíam `zona_classificacao` preenchida. Esses casos correspondem aos registros cuja pontuação inválida não pôde ser recalculada devido à ausência do `eixo2_total`.
- Como a classificação depende diretamente da pontuação final, a `zona_classificacao` desses registros foi definida como ausente (`NaN`), evitando a manutenção de uma classificação sem pontuação válida que a sustente.

In [47]:
def classificar_zona(pontuacao):
    if pd.isna(pontuacao):
        return np.nan
    elif pontuacao >= 850:
        return "Excelência"
    elif pontuacao >= 700:
        return "Qualidade"
    elif pontuacao >= 550:
        return "Aperfeiçoamento"
    elif pontuacao >= 400:
        return "Crítica"
    else:
        return "Insuficiente"

In [48]:
resultados_tratado["zona_esperada"] = (
    resultados_tratado["pontuacao_final"]
    .apply(classificar_zona)
)

inconsistencia_zona = (
    resultados_tratado["pontuacao_final"].notna()
    & (
        resultados_tratado["zona_classificacao"]
        != resultados_tratado["zona_esperada"]
    )
)

print(
    "Classificações incompatíveis com a pontuação:",
    inconsistencia_zona.sum()
)

Classificações incompatíveis com a pontuação: 0


In [49]:
resultados_tratado.loc[
    inconsistencia_zona,
    [
        "codigo_ine",
        "ciclo",
        "pontuacao_final",
        "zona_classificacao",
        "zona_esperada"
    ]
]

,codigo_ine,ciclo,pontuacao_final,zona_classificacao,zona_esperada


In [50]:
sem_pontuacao_com_zona = (
    resultados_tratado["pontuacao_final"].isna()
    & resultados_tratado["zona_classificacao"].notna()
)

print(
    "Registros sem pontuação final, mas com classificação:",
    sem_pontuacao_com_zona.sum()
)

resultados_tratado.loc[
    sem_pontuacao_com_zona,
    [
        "codigo_ine",
        "ciclo",
        "pontuacao_final",
        "zona_classificacao",
        "situacao_participacao"
    ]
]

Registros sem pontuação final, mas com classificação: 2


,codigo_ine,ciclo,pontuacao_final,zona_classificacao,situacao_participacao
1201,6000000028,2023.3,NaN,Aperfeiçoamento,Participante
1335,6000000110,2023.3,NaN,Aperfeiçoamento,Participante


In [53]:
resultados_tratado.loc[
    sem_pontuacao_com_zona,
    "zona_classificacao"
] = np.nan

print(
    "Registros sem pontuação final, mas com classificação:",
    (
        resultados_tratado["pontuacao_final"].isna()
        & resultados_tratado["zona_classificacao"].notna()
    ).sum()
)

Registros sem pontuação final, mas com classificação: 0


In [54]:
resultados_tratado = resultados_tratado.drop(
    columns=[
        "pontuacao_calculada",
        "diferenca_pontuacao",
        "zona_esperada"
    ]
)

In [55]:
resultados_tratado.columns

Index(['codigo_ine', 'ciclo', 'eixo1_bloco_a', 'eixo1_bloco_b',
       'eixo1_bloco_c', 'eixo1_total', 'taxa_resposta_bloco_a',
       'bloco_a_valido', 'eixo2_total', 'eixo3_total', 'pontuacao_final',
       'zona_classificacao', 'situacao_participacao'],
      dtype='str')

### 2.9 Valores ausentes em `eixo2_total`

- Durante o entendimento dos dados foi identificada uma quantidade relevante de valores ausentes em `eixo2_total`.
- Como a ausência de pontuação pode possuir significados diferentes conforme a situação de participação da equipe, os registros serão avaliados separadamente entre participantes e desclassificados.
- Valores ausentes não serão automaticamente substituídos por zero, média ou mediana, evitando a criação artificial de pontuações.

In [56]:
print(
    "Total de valores ausentes em eixo2_total:",
    resultados_tratado["eixo2_total"].isna().sum()
)

print("\nAusentes por situação de participação:")

print(
    resultados_tratado.loc[
        resultados_tratado["eixo2_total"].isna(),
        "situacao_participacao"
    ].value_counts(dropna=False)
)

Total de valores ausentes em eixo2_total: 227

Ausentes por situação de participação:
situacao_participacao
Participante       169
Desclassificada     58
Name: count, dtype: int64


#### Decisão de tratamento

- Após os tratamentos anteriores, foram identificados 227 valores ausentes em `eixo2_total`, sendo 58 referentes a registros desclassificados e 169 a participantes.
- Nos registros desclassificados, a ausência de pontuação é compatível com a regra de negócio, uma vez que equipes desclassificadas não possuem pontuação final.
- Nos 169 registros de participantes, a ausência de `eixo2_total` foi preservada. Não há informação suficiente para reconstruir essa pontuação de forma confiável, e sua substituição por zero, média ou mediana poderia introduzir valores artificiais e distorcer as análises.
- Dessa forma, os valores ausentes foram mantidos e documentados como uma limitação de completude da base.

### 2.10 Validação final da tabela `resultados_ciclos`

- Após as etapas de padronização, conversão de tipos, tratamento de duplicidades e validação das regras de negócio, são realizadas verificações finais para avaliar a consistência da tabela preparada.

In [57]:
print("Dimensão final:", resultados_tratado.shape)

print(
    "\nChaves duplicadas:",
    resultados_tratado.duplicated(
        subset=["codigo_ine", "ciclo"]
    ).sum()
)

print(
    "Ciclos distintos:",
    resultados_tratado["ciclo"].nunique()
)

print(
    "Taxas fora de 0–1:",
    (
        resultados_tratado["taxa_resposta_bloco_a"].notna()
        & ~resultados_tratado["taxa_resposta_bloco_a"].between(0, 1)
    ).sum()
)

print(
    "Pontuações finais fora de 0–1000:",
    (
        resultados_tratado["pontuacao_final"].notna()
        & ~resultados_tratado["pontuacao_final"].between(0, 1000)
    ).sum()
)

print(
    "Registros sem pontuação, mas com classificação:",
    (
        resultados_tratado["pontuacao_final"].isna()
        & resultados_tratado["zona_classificacao"].notna()
    ).sum()
)

print(
    "Ausentes em eixo2_total:",
    resultados_tratado["eixo2_total"].isna().sum()
)

print(
    "Ausentes em pontuacao_final:",
    resultados_tratado["pontuacao_final"].isna().sum()
)

Dimensão final: (1376, 13)

Chaves duplicadas: 0
Ciclos distintos: 11
Taxas fora de 0–1: 0
Pontuações finais fora de 0–1000: 0
Registros sem pontuação, mas com classificação: 0
Ausentes em eixo2_total: 227
Ausentes em pontuacao_final: 60


## 3. Preparação da tabela `eixo3_indicadores`

### 3.1 Padronização da variável `ciclo`

- A variável `ciclo` foi interpretada como `float64`, embora represente um identificador temporal e não uma medida numérica.
- Para manter consistência com a tabela `resultados_ciclos` e facilitar os relacionamentos entre as tabelas, os ciclos serão convertidos para texto no formato `AAAA.Q`.

In [58]:
print("Tipo antes:",
      eixo3_tratado["ciclo"].dtype)

print("Quantidade de ciclos:",
      eixo3_tratado["ciclo"].nunique())

print("\nValores encontrados:")
print(
    eixo3_tratado["ciclo"]
    .value_counts()
    .sort_index()
)

eixo3_tratado["ciclo"] = (
    eixo3_tratado["ciclo"]
    .map(lambda x: f"{x:.1f}")
)

Tipo antes: float64
Quantidade de ciclos: 11

Valores encontrados:
ciclo
2023.1    1026
2023.2    1084
2023.3    1104
2024.1    1184
2024.2    1220
2024.3    1246
2025.1    1286
2025.2    1286
2025.3    1286
2026.1    1286
2026.2    1286
Name: count, dtype: int64


In [59]:
print("Tipo depois:",
      eixo3_tratado["ciclo"].dtype)

print("Quantidade de ciclos:",
      eixo3_tratado["ciclo"].nunique())

print(
    "Ciclos diferentes dos esperados:",
    set(eixo3_tratado["ciclo"].unique()) - ciclos_esperados
)

print(
    "Ciclos esperados ausentes:",
    ciclos_esperados - set(eixo3_tratado["ciclo"].unique())
)

Tipo depois: str
Quantidade de ciclos: 11
Ciclos diferentes dos esperados: set()
Ciclos esperados ausentes: set()


### 3.2 Padronização de `percentual_alcancado`

- Durante o entendimento dos dados, foi identificada uma mistura de escalas na variável `percentual_alcancado`.
- Parte dos registros utiliza valores entre 0 e 1, enquanto outros utilizam valores superiores a 1 e até 100, indicando representação percentual na escala de 0 a 100.
- Para garantir consistência nos cálculos e análises posteriores, a variável será padronizada para a escala de 0 a 1. Valores superiores a 1 serão divididos por 100.
- A consistência dessa transformação será posteriormente validada por meio da relação entre `percentual_alcancado`, `peso_maximo` e `pontos_obtidos`.

In [60]:
print(
    "Mínimo:",
    eixo3_tratado["percentual_alcancado"].min()
)

print(
    "Máximo:",
    eixo3_tratado["percentual_alcancado"].max()
)

print(
    "Valores entre 0 e 1:",
    eixo3_tratado["percentual_alcancado"]
    .between(0, 1)
    .sum()
)

print(
    "Valores maiores que 1:",
    (
        eixo3_tratado["percentual_alcancado"] > 1
    ).sum()
)

print(
    "Valores fora de 0–100:",
    (
        ~eixo3_tratado["percentual_alcancado"]
        .between(0, 100)
    ).sum()
)

Mínimo: 0.05
Máximo: 100.0
Valores entre 0 e 1: 12914
Valores maiores que 1: 380
Valores fora de 0–100: 0


In [61]:
escala_percentual_100 = (
    eixo3_tratado["percentual_alcancado"] > 1
)

eixo3_tratado.loc[
    escala_percentual_100,
    "percentual_alcancado"
] = (
    eixo3_tratado.loc[
        escala_percentual_100,
        "percentual_alcancado"
    ] / 100
)

print(
    "Mínimo após padronização:",
    eixo3_tratado["percentual_alcancado"].min()
)

print(
    "Máximo após padronização:",
    eixo3_tratado["percentual_alcancado"].max()
)

print(
    "Valores fora de 0–1:",
    (
        ~eixo3_tratado["percentual_alcancado"]
        .between(0, 1)
    ).sum()
)

Mínimo após padronização: 0.05
Máximo após padronização: 1.0
Valores fora de 0–1: 0


### 3.3 Validação da pontuação dos indicadores

- Após a padronização de `percentual_alcancado` para a escala de 0 a 1, foi verificada a consistência entre o percentual alcançado, o peso máximo do indicador e os pontos obtidos.
- Pela regra de cálculo, espera-se que `pontos_obtidos` corresponda aproximadamente ao produto entre `peso_maximo` e `percentual_alcancado`. Pequenas diferenças decorrentes de arredondamento são consideradas aceitáveis.

In [62]:
pontos_esperados = (
    eixo3_tratado["peso_maximo"]
    * eixo3_tratado["percentual_alcancado"]
)

diferenca_pontos = (
    eixo3_tratado["pontos_obtidos"]
    - pontos_esperados
).abs()

print(
    "Maior diferença encontrada:",
    diferenca_pontos.max()
)

print(
    "Registros com diferença superior a 0.15:",
    (diferenca_pontos > 0.15).sum()
)

Maior diferença encontrada: 0.054000000000002046
Registros com diferença superior a 0.15: 0


#### Resultado da validação

- Após a padronização de `percentual_alcancado`, a maior diferença observada entre os pontos registrados e os pontos calculados pela regra `peso_maximo × percentual_alcancado` foi de aproximadamente 0,054 ponto.
- Nenhum registro apresentou diferença superior à tolerância de 0,15 ponto. O resultado indica que a padronização para a escala de 0 a 1 preservou a consistência entre o percentual alcançado e os pontos obtidos.
- As pequenas diferenças observadas são compatíveis com arredondamentos dos valores armazenados.

### 3.4 Validação da soma dos pesos por equipe e ciclo

- Os indicadores aplicáveis no Eixo 3 podem variar de acordo com o grupo da equipe. Dessa forma, a quantidade e a composição dos indicadores não precisam ser iguais para todas as equipes.
- Entretanto, conforme a regra de negócio, a soma dos pesos máximos dos indicadores aplicáveis deve totalizar 600 pontos para cada combinação de equipe e ciclo.

In [63]:
soma_pesos = (
    eixo3_tratado
    .groupby(["codigo_ine", "ciclo"])["peso_maximo"]
    .sum()
)

print(
    "Combinações equipe × ciclo:",
    len(soma_pesos)
)

print(
    "Soma mínima dos pesos:",
    soma_pesos.min()
)

print(
    "Soma máxima dos pesos:",
    soma_pesos.max()
)

print(
    "Combinações com soma diferente de 600:",
    (soma_pesos != 600).sum()
)

Combinações equipe × ciclo: 1363
Soma mínima dos pesos: 600
Soma máxima dos pesos: 600
Combinações com soma diferente de 600: 0


#### Resultado da validação

- Foram identificadas 1.363 combinações de equipe e ciclo na tabela. Em todas elas a soma dos pesos máximos dos indicadores foi igual a 600 pontos.
- Dessa forma, a regra de negócio referente à distribuição dos pesos do Eixo 3 foi integralmente atendida, não sendo necessário realizar tratamento adicional.

### 3.5 Validação final da tabela `eixo3_indicadores`

- Após a padronização do ciclo e da escala de `percentual_alcancado`, são realizadas verificações finais para confirmar a consistência estrutural e das principais regras de negócio da tabela.

In [64]:
print("Dimensão final:", eixo3_tratado.shape)

print(
    "\nChaves duplicadas:",
    eixo3_tratado.duplicated(
        subset=["codigo_ine", "ciclo", "codigo_indicador"]
    ).sum()
)

print(
    "Ciclos distintos:",
    eixo3_tratado["ciclo"].nunique()
)

print(
    "Indicadores distintos:",
    eixo3_tratado["codigo_indicador"].nunique()
)

print(
    "Valores ausentes:",
    eixo3_tratado.isna().sum().sum()
)

print(
    "Percentuais fora de 0–1:",
    (
        ~eixo3_tratado["percentual_alcancado"]
        .between(0, 1)
    ).sum()
)

print(
    "Inconsistências no cálculo dos pontos:",
    (diferenca_pontos > 0.15).sum()
)

print(
    "Combinações equipe × ciclo com pesos diferentes de 600:",
    (soma_pesos != 600).sum()
)

Dimensão final: (13294, 7)

Chaves duplicadas: 0
Ciclos distintos: 11
Indicadores distintos: 10
Valores ausentes: 0
Percentuais fora de 0–1: 0
Inconsistências no cálculo dos pontos: 0
Combinações equipe × ciclo com pesos diferentes de 600: 0


## 4. Preparação da tabela `nps_atributos`

### 4.1 Padronização da variável `ciclo`

- Assim como em `eixo3_indicadores`, a variável `ciclo` foi interpretada como numérica, embora represente um identificador temporal.
- Para manter consistência entre as tabelas e facilitar os relacionamentos posteriores, os ciclos serão convertidos para texto no formato `AAAA.Q`.

In [65]:
print("Tipo antes:", nps_tratado["ciclo"].dtype)

print(
    "Quantidade de ciclos:",
    nps_tratado["ciclo"].nunique()
)

print("\nValores encontrados:")
print(
    nps_tratado["ciclo"]
    .value_counts()
    .sort_index()
)

Tipo antes: float64
Quantidade de ciclos: 11

Valores encontrados:
ciclo
2023.1    525
2023.2    555
2023.3    565
2024.1    605
2024.2    625
2024.3    640
2025.1    660
2025.2    660
2025.3    660
2026.1    660
2026.2    660
Name: count, dtype: int64


In [66]:
nps_tratado["ciclo"] = (
    nps_tratado["ciclo"]
    .map(lambda x: f"{x:.1f}")
)

print("Tipo depois:", nps_tratado["ciclo"].dtype)

print(
    "Quantidade de ciclos:",
    nps_tratado["ciclo"].nunique()
)

print(
    "Ciclos diferentes dos esperados:",
    set(nps_tratado["ciclo"].unique()) - ciclos_esperados
)

print(
    "Ciclos esperados ausentes:",
    ciclos_esperados - set(nps_tratado["ciclo"].unique())
)

Tipo depois: str
Quantidade de ciclos: 11
Ciclos diferentes dos esperados: set()
Ciclos esperados ausentes: set()


### 4.2 Valores ausentes em `nps`

- Durante o entendimento dos dados, foram identificados valores ausentes na variável `nps`.
- Como a tabela também informa a quantidade de respondentes, será verificado se a ausência do NPS está associada à inexistência de respostas ou se também ocorre em registros com respondentes.
- Essa análise é necessária antes de qualquer decisão de tratamento, evitando a substituição de valores ausentes sem evidência suficiente.

In [67]:
nps_ausente = nps_tratado["nps"].isna()

print(
    "Total de NPS ausentes:",
    nps_ausente.sum()
)

print(
    "NPS ausentes com zero respondentes:",
    (
        nps_ausente
        & (nps_tratado["n_respondentes"] == 0)
    ).sum()
)

print(
    "NPS ausentes com respondentes:",
    (
        nps_ausente
        & (nps_tratado["n_respondentes"] > 0)
    ).sum()
)

Total de NPS ausentes: 260
NPS ausentes com zero respondentes: 0
NPS ausentes com respondentes: 260


In [68]:
nps_tratado.loc[
    nps_ausente,
    "n_respondentes"
].describe()

count    260.000000
mean      42.596154
std       24.464030
min        3.000000
25%       25.000000
50%       40.000000
75%       57.000000
max      127.000000
Name: n_respondentes, dtype: float64

#### Decisão de tratamento

- Foram identificados 260 valores ausentes em `nps`. Todos esses registros possuem respondentes, com quantidade variando entre 3 e 127 pessoas.
- Dessa forma, a ausência do NPS não pode ser atribuída à inexistência de respostas. Como a base não disponibiliza as respostas individuais necessárias para recalcular o indicador, não há evidência suficiente para reconstruir esses valores.
- Os valores ausentes foram preservados. Não foi realizada substituição por zero, média ou mediana, pois isso poderia introduzir valores artificiais e distorcer a interpretação do indicador.

### 4.3 Validação dos domínios

- São verificadas as faixas válidas das variáveis `nps` e `n_respondentes`, preservando os valores ausentes de NPS já documentados anteriormente.

In [69]:
print(
    "NPS fora de -100 a 100:",
    (
        nps_tratado["nps"].notna()
        & ~nps_tratado["nps"].between(-100, 100)
    ).sum()
)

print(
    "Número de respondentes negativo:",
    (
        nps_tratado["n_respondentes"] < 0
    ).sum()
)

print(
    "NPS mínimo:",
    nps_tratado["nps"].min()
)

print(
    "NPS máximo:",
    nps_tratado["nps"].max()
)

print(
    "Número mínimo de respondentes:",
    nps_tratado["n_respondentes"].min()
)

print(
    "Número máximo de respondentes:",
    nps_tratado["n_respondentes"].max()
)

NPS fora de -100 a 100: 0
Número de respondentes negativo: 0
NPS mínimo: -70.0
NPS máximo: 100.0
Número mínimo de respondentes: 3
Número máximo de respondentes: 146


#### Resultado da validação

- Os valores existentes de NPS encontram-se integralmente no intervalo esperado de -100 a 100, e não foram identificadas quantidades negativas de respondentes.
- A quantidade de respondentes varia entre 3 e 146. Portanto, não foi necessário realizar tratamento adicional nos domínios dessas variáveis.

### 4.4 Validação da chave e dos atributos por equipe e ciclo

- A chave esperada da tabela é composta por `codigo_ine`, `ciclo` e `codigo_atributo`.
- Além da unicidade dessa chave, será verificado se cada combinação de equipe e ciclo possui os cinco atributos de NPS previstos na estrutura da base.

In [70]:
print(
    "Chaves duplicadas:",
    nps_tratado.duplicated(
        subset=["codigo_ine", "ciclo", "codigo_atributo"]
    ).sum()
)

atributos_por_equipe_ciclo = (
    nps_tratado
    .groupby(["codigo_ine", "ciclo"])["codigo_atributo"]
    .nunique()
)

print(
    "Combinações equipe × ciclo:",
    len(atributos_por_equipe_ciclo)
)

print(
    "Mínimo de atributos por combinação:",
    atributos_por_equipe_ciclo.min()
)

print(
    "Máximo de atributos por combinação:",
    atributos_por_equipe_ciclo.max()
)

print(
    "Combinações com quantidade diferente de 5:",
    (atributos_por_equipe_ciclo != 5).sum()
)

Chaves duplicadas: 0
Combinações equipe × ciclo: 1363
Mínimo de atributos por combinação: 5
Máximo de atributos por combinação: 5
Combinações com quantidade diferente de 5: 0


#### Resultado da validação

- Não foram identificadas duplicidades na chave composta por `codigo_ine`, `ciclo` e `codigo_atributo`.
- Foram encontradas 1.363 combinações de equipe e ciclo, e todas possuem exatamente os cinco atributos de NPS previstos na estrutura da base. Portanto, não foi necessário realizar tratamento adicional nesse aspecto.

### 4.5 Validação final da tabela `nps_atributos`

- Após a padronização do ciclo e a análise dos valores ausentes, são realizadas verificações finais para confirmar a consistência estrutural e dos domínios da tabela.

In [71]:
print("Dimensão final:", nps_tratado.shape)

print(
    "\nChaves duplicadas:",
    nps_tratado.duplicated(
        subset=["codigo_ine", "ciclo", "codigo_atributo"]
    ).sum()
)

print(
    "Ciclos distintos:",
    nps_tratado["ciclo"].nunique()
)

print(
    "Atributos distintos:",
    nps_tratado["codigo_atributo"].nunique()
)

print(
    "NPS ausentes:",
    nps_tratado["nps"].isna().sum()
)

print(
    "NPS fora de -100 a 100:",
    (
        nps_tratado["nps"].notna()
        & ~nps_tratado["nps"].between(-100, 100)
    ).sum()
)

print(
    "Respondentes negativos:",
    (nps_tratado["n_respondentes"] < 0).sum()
)

print(
    "Combinações equipe × ciclo com quantidade diferente de 5 atributos:",
    (atributos_por_equipe_ciclo != 5).sum()
)

Dimensão final: (6815, 6)

Chaves duplicadas: 0
Ciclos distintos: 11
Atributos distintos: 5
NPS ausentes: 260
NPS fora de -100 a 100: 0
Respondentes negativos: 0
Combinações equipe × ciclo com quantidade diferente de 5 atributos: 0


## 5. Preparação da tabela `planos_acao`

### 5.1 Padronização da variável `ciclo`

- A variável `ciclo` será padronizada para texto no formato `AAAA.Q`, mantendo o mesmo padrão adotado nas demais tabelas do projeto.
- Essa padronização facilita os relacionamentos entre as bases e evita diferenças de tipo entre campos que representam o mesmo conceito.

In [72]:
print("Tipo antes:", planos_tratado["ciclo"].dtype)

print(
    "Quantidade de ciclos:",
    planos_tratado["ciclo"].nunique()
)

print("\nValores encontrados:")
print(
    planos_tratado["ciclo"]
    .value_counts()
    .sort_index()
)

Tipo antes: float64
Quantidade de ciclos: 11

Valores encontrados:
ciclo
2023.1    33
2023.2    39
2023.3    34
2024.1    30
2024.2    38
2024.3    35
2025.1    40
2025.2    43
2025.3    36
2026.1    50
2026.2    42
Name: count, dtype: int64


In [73]:
planos_tratado["ciclo"] = (
    planos_tratado["ciclo"]
    .map(lambda x: f"{x:.1f}")
)

print("Tipo depois:", planos_tratado["ciclo"].dtype)

print(
    "Quantidade de ciclos:",
    planos_tratado["ciclo"].nunique()
)

print(
    "Ciclos diferentes dos esperados:",
    set(planos_tratado["ciclo"].unique()) - ciclos_esperados
)

print(
    "Ciclos esperados ausentes:",
    ciclos_esperados - set(planos_tratado["ciclo"].unique())
)

Tipo depois: str
Quantidade de ciclos: 11
Ciclos diferentes dos esperados: set()
Ciclos esperados ausentes: set()


### 5.2 Valores ausentes em `texto_plano`

- Durante o entendimento dos dados, foram identificados registros sem conteúdo na variável `texto_plano`.
- Como o texto representa uma informação qualitativa que não pode ser reconstruída de forma confiável a partir das demais variáveis, os registros serão avaliados antes da definição do tratamento.

In [74]:
texto_ausente = planos_tratado["texto_plano"].isna()

print(
    "Textos de plano ausentes:",
    texto_ausente.sum()
)

print(
    "Registros com texto ausente e id_plano preenchido:",
    (
        texto_ausente
        & planos_tratado["id_plano"].notna()
    ).sum()
)

print(
    "Registros com texto ausente e codigo_ine preenchido:",
    (
        texto_ausente
        & planos_tratado["codigo_ine"].notna()
    ).sum()
)

print(
    "Registros com texto ausente e ciclo preenchido:",
    (
        texto_ausente
        & planos_tratado["ciclo"].notna()
    ).sum()
)

Textos de plano ausentes: 10
Registros com texto ausente e id_plano preenchido: 10
Registros com texto ausente e codigo_ine preenchido: 10
Registros com texto ausente e ciclo preenchido: 10


#### Decisão de tratamento

- Foram identificados 10 registros com `texto_plano` ausente. Todos possuem `id_plano`, `codigo_ine` e `ciclo` preenchidos, indicando que se tratam de registros estruturalmente válidos.
- Como o conteúdo textual do plano não pode ser reconstruído de forma confiável a partir das demais variáveis, os valores ausentes foram preservados.
- As linhas não foram excluídas, pois ainda contêm informações estruturadas relevantes para a análise, como as dimensões de avaliação e o nível de maturidade.

### 5.3 Validação das dimensões de avaliação
- As variáveis de avaliação dos planos de ação serão verificadas em relação aos domínios definidos no dicionário de dados.
- As dimensões de governança, SMART e viabilidade devem apresentar valores entre 1 e 5, enquanto `nivel_maturidade` deve apresentar valores entre 1 e 3.
- A ausência de determinados valores possíveis dentro desses intervalos não representa, por si só, uma inconsistência.

In [75]:
colunas_dimensoes = [
    "dim_governanca",
    "dim_smart",
    "dim_viabilidade"
]

for coluna in colunas_dimensoes:
    print(f"\n{coluna}")
    print("Mínimo:", planos_tratado[coluna].min())
    print("Máximo:", planos_tratado[coluna].max())
    print(
        "Fora de 1–5:",
        (
            planos_tratado[coluna].notna()
            & ~planos_tratado[coluna].between(1, 5)
        ).sum()
    )

print("\nnivel_maturidade")
print("Mínimo:", planos_tratado["nivel_maturidade"].min())
print("Máximo:", planos_tratado["nivel_maturidade"].max())
print(
    "Fora de 1–3:",
    (
        planos_tratado["nivel_maturidade"].notna()
        & ~planos_tratado["nivel_maturidade"].between(1, 3)
    ).sum()
)


dim_governanca
Mínimo: 1
Máximo: 4
Fora de 1–5: 0

dim_smart
Mínimo: 1
Máximo: 5
Fora de 1–5: 0

dim_viabilidade
Mínimo: 1
Máximo: 4
Fora de 1–5: 0

nivel_maturidade
Mínimo: 1
Máximo: 3
Fora de 1–3: 0


#### Resultado da validação

- Todas as dimensões de avaliação apresentaram valores dentro dos domínios definidos no dicionário de dados.
- As dimensões `dim_governanca` e `dim_viabilidade` apresentaram valores entre 1 e 4, enquanto `dim_smart` apresentou valores entre 1 e 5. O `nivel_maturidade` apresentou valores entre 1 e 3.
- Não foram identificados valores fora dos intervalos permitidos, portanto não foi necessário realizar tratamento adicional.

### 5.4 Consistência entre nível e rótulo de maturidade

- O `nivel_maturidade` possui uma correspondência esperada com `rotulo_maturidade`. Será verificado se os registros respeitam a seguinte relação:

    - nível 1: Reformulação;
    - nível 2: Ajustes;
    - nível 3: Aprovado.

- Essa validação permite identificar inconsistências entre duas variáveis que, isoladamente, poderiam apresentar valores válidos.

In [77]:
mapa_maturidade = {
    1: "Reformulação",
    2: "Ajustes",
    3: "Aprovado"
}

rotulo_esperado = (
    planos_tratado["nivel_maturidade"]
    .map(mapa_maturidade)
)

inconsistencia_maturidade = (
    planos_tratado["rotulo_maturidade"]
    != rotulo_esperado
)

print(
    "Registros com nível e rótulo incompatíveis:",
    inconsistencia_maturidade.sum()
)

Registros com nível e rótulo incompatíveis: 0


In [78]:
pd.crosstab(
    planos_tratado["nivel_maturidade"],
    planos_tratado["rotulo_maturidade"]
)

rotulo_maturidade,Ajustes,Aprovado,Reformulação
nivel_maturidade,,,
1,0,0,122
2,170,0,0
3,0,128,0


#### Resultado da validação

- A relação entre `nivel_maturidade` e `rotulo_maturidade` mostrou-se integralmente consistente.
- Os 122 registros de nível 1 estão classificados como "Reformulação", os 170 registros de nível 2 como "Ajustes" e os 128 registros de nível 3 como "Aprovado".
- Não foram identificadas combinações incompatíveis entre nível e rótulo de maturidade, portanto não foi necessário realizar tratamento adicional.

### 5.5 Validação final da tabela `planos_acao`

- Após a padronização do ciclo e a análise dos valores ausentes, dos domínios das dimensões e da correspondência entre nível e rótulo de maturidade, são realizadas verificações finais da tabela preparada.

In [79]:
print("Dimensão final:", planos_tratado.shape)

print(
    "\nIDs de plano duplicados:",
    planos_tratado["id_plano"].duplicated().sum()
)

print(
    "Ciclos distintos:",
    planos_tratado["ciclo"].nunique()
)

print(
    "Textos de plano ausentes:",
    planos_tratado["texto_plano"].isna().sum()
)

print(
    "Dimensões fora de 1–5:",
    (
        (~planos_tratado["dim_governanca"].between(1, 5))
        | (~planos_tratado["dim_smart"].between(1, 5))
        | (~planos_tratado["dim_viabilidade"].between(1, 5))
    ).sum()
)

print(
    "Níveis de maturidade fora de 1–3:",
    (~planos_tratado["nivel_maturidade"].between(1, 3)).sum()
)

print(
    "Inconsistências entre nível e rótulo:",
    inconsistencia_maturidade.sum()
)

Dimensão final: (420, 9)

IDs de plano duplicados: 0
Ciclos distintos: 11
Textos de plano ausentes: 10
Dimensões fora de 1–5: 0
Níveis de maturidade fora de 1–3: 0
Inconsistências entre nível e rótulo: 0


## 6. Validação integrada das tabelas

- Após a preparação individual das cinco tabelas, são realizadas validações integradas para verificar a consistência dos relacionamentos entre elas.
- As verificações consideram principalmente os identificadores de equipe (`codigo_ine`), os ciclos de monitoramento e as combinações entre equipe e ciclo.

### 6.1 Integridade referencial de `codigo_ine`

- A tabela `equipes` é utilizada como referência cadastral das equipes. Dessa forma, os códigos de equipe presentes nas demais tabelas devem possuir correspondência em `equipes`.

In [80]:
ines_equipes = set(equipes_tratado["codigo_ine"])

tabelas_validacao = {
    "resultados_ciclos": resultados_tratado,
    "eixo3_indicadores": eixo3_tratado,
    "nps_atributos": nps_tratado,
    "planos_acao": planos_tratado
}

for nome, df in tabelas_validacao.items():
    ines_tabela = set(df["codigo_ine"])
    sem_correspondencia = ines_tabela - ines_equipes

    print(f"\n{nome}")
    print("INEs distintos:", len(ines_tabela))
    print(
        "INEs sem correspondência em equipes:",
        len(sem_correspondencia)
    )


resultados_ciclos
INEs distintos: 145
INEs sem correspondência em equipes: 13

eixo3_indicadores
INEs distintos: 132
INEs sem correspondência em equipes: 0

nps_atributos
INEs distintos: 132
INEs sem correspondência em equipes: 0

planos_acao
INEs distintos: 128
INEs sem correspondência em equipes: 0


#### Investigação dos INEs sem correspondência

- A validação referencial identificou 13 códigos de equipe presentes em `resultados_ciclos` que não possuem correspondência na tabela cadastral `equipes`.
- Como a exclusão automática desses registros poderia eliminar observações válidas, os casos serão investigados antes da definição do tratamento.

In [81]:
ines_resultados_sem_cadastro = (
    set(resultados_tratado["codigo_ine"])
    - set(equipes_tratado["codigo_ine"])
)

print(
    "Quantidade de INEs sem cadastro:",
    len(ines_resultados_sem_cadastro)
)

print(
    "INEs encontrados:",
    sorted(ines_resultados_sem_cadastro)
)

Quantidade de INEs sem cadastro: 13
INEs encontrados: ['7109272546', '7250817686', '7304288101', '7306314498', '7355030972', '7530568026', '7532367409', '7624579721', '7676874649', '7868338244', '7913642309', '7917583207', '7949616694']


In [82]:
resultados_sem_cadastro = resultados_tratado[
    resultados_tratado["codigo_ine"]
    .isin(ines_resultados_sem_cadastro)
]

print(
    "Quantidade de registros envolvidos:",
    len(resultados_sem_cadastro)
)

print("\nRegistros por INE:")

print(
    resultados_sem_cadastro["codigo_ine"]
    .value_counts()
    .sort_index()
)

Quantidade de registros envolvidos: 13

Registros por INE:
codigo_ine
7109272546    1
7250817686    1
7304288101    1
7306314498    1
7355030972    1
7530568026    1
7532367409    1
7624579721    1
7676874649    1
7868338244    1
7913642309    1
7917583207    1
7949616694    1
Name: count, dtype: int64


In [83]:
resultados_sem_cadastro[
    [
        "codigo_ine",
        "ciclo",
        "situacao_participacao",
        "eixo1_total",
        "eixo2_total",
        "eixo3_total",
        "pontuacao_final"
    ]
].sort_values(["ciclo", "codigo_ine"])

,codigo_ine,ciclo,situacao_participacao,eixo1_total,eixo2_total,eixo3_total,pontuacao_final
1374,7304288101,2023.1,Participante,128.1,NaN,407.2,693.3
584,7532367409,2023.1,Participante,138.3,130.8,471.2,740.3
853,7306314498,2024.1,Participante,125.2,145.7,388.1,659.0
509,7530568026,2024.3,Participante,133.6,126.4,411.5,671.5
115,7868338244,2024.3,Participante,112.5,98.9,263.2,474.6
307,7949616694,2024.3,Participante,148.1,115.1,393.6,656.8
1119,7917583207,2025.1,Participante,148.9,180.5,486.0,815.4
42,7676874649,2025.2,Participante,79.8,83.5,294.9,458.2
33,7109272546,2026.1,Participante,122.1,80.7,370.0,572.8
78,7913642309,2026.1,Participante,126.4,90.5,329.3,546.2


#### Tratamento dos registros sem correspondência cadastral

- Foram identificados 13 registros em `resultados_ciclos` cujos códigos `codigo_ine` não possuem correspondência na tabela `equipes`.
- Cada código ocorre apenas uma vez na tabela de resultados, entre os registros identificados, 12 correspondem a participantes e um a uma equipe desclassificada.
- Como `equipes` constitui a referência cadastral utilizada para caracterizar as equipes e os 13 códigos também não possuem correspondência nas demais tabelas analíticas, esses registros foram considerados órfãos.
- Para preservar a integridade referencial da base preparada e evitar análises de resultados associados a equipes sem cadastro correspondente, os 13 registros foram removidos de `resultados_ciclos`.

In [84]:
resultados_tratado = (
    resultados_tratado[
        ~resultados_tratado["codigo_ine"]
        .isin(ines_resultados_sem_cadastro)
    ]
    .reset_index(drop=True)
)

In [85]:
ines_sem_correspondencia = (
    set(resultados_tratado["codigo_ine"])
    - set(equipes_tratado["codigo_ine"])
)

print("Dimensão após tratamento:", resultados_tratado.shape)

print(
    "INEs distintos em resultados:",
    resultados_tratado["codigo_ine"].nunique()
)

print(
    "INEs sem correspondência em equipes:",
    len(ines_sem_correspondencia)
)

print(
    "Chaves equipe × ciclo duplicadas:",
    resultados_tratado.duplicated(
        subset=["codigo_ine", "ciclo"]
    ).sum()
)

Dimensão após tratamento: (1363, 13)
INEs distintos em resultados: 132
INEs sem correspondência em equipes: 0
Chaves equipe × ciclo duplicadas: 0


### 6.2 Consistência das combinações entre equipe e ciclo

- Após o tratamento da integridade referencial, `resultados_ciclos`, `eixo3_indicadores` e `nps_atributos` apresentam 1.363 combinações de equipe e ciclo.
- Entretanto, a igualdade na quantidade de combinações não garante que sejam exatamente os mesmos registros. Por isso, serão comparados os conjuntos formados por `codigo_ine` e `ciclo` nas três tabelas.

In [88]:
pares_resultados = set(
    resultados_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

pares_eixo3 = set(
    eixo3_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

pares_nps = set(
    nps_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

print("Combinações em resultados:", len(pares_resultados))
print("Combinações em eixo3:", len(pares_eixo3))
print("Combinações em NPS:", len(pares_nps))

print(
    "\nResultados que não aparecem no Eixo 3:",
    len(pares_resultados - pares_eixo3)
)

print(
    "Eixo 3 que não aparece em resultados:",
    len(pares_eixo3 - pares_resultados)
)

Combinações em resultados: 1363
Combinações em eixo3: 1363
Combinações em NPS: 1363

Resultados que não aparecem no Eixo 3: 0
Eixo 3 que não aparece em resultados: 0


In [89]:
print(
    "Resultados que não aparecem no NPS:",
    len(pares_resultados - pares_nps)
)

print(
    "NPS que não aparece em resultados:",
    len(pares_nps - pares_resultados)
)

print(
    "Eixo 3 que não aparece no NPS:",
    len(pares_eixo3 - pares_nps)
)

print(
    "NPS que não aparece no Eixo 3:",
    len(pares_nps - pares_eixo3)
)

Resultados que não aparecem no NPS: 0
NPS que não aparece em resultados: 0
Eixo 3 que não aparece no NPS: 0
NPS que não aparece no Eixo 3: 0


#### Resultado da validação

- As tabelas `resultados_ciclos`, `eixo3_indicadores` e `nps_atributos` apresentam exatamente as mesmas 1.363 combinações de `codigo_ine` e `ciclo`.
- Não foram identificadas combinações presentes em uma dessas tabelas e ausentes nas demais. Dessa forma, foi confirmada a consistência das chaves de equipe e ciclo entre as três bases analíticas.

### 6.3 Integridade das combinações de `planos_acao`

- Diferentemente das demais tabelas analíticas, `planos_acao` não precisa possuir um registro para cada combinação de equipe e ciclo.
- Portanto, a validação não exige igualdade de cobertura. O objetivo é verificar se todas as combinações de equipe e ciclo presentes nos planos de ação possuem correspondência nas demais tabelas preparadas.

In [90]:
pares_planos = set(
    planos_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

print(
    "Registros em planos_acao:",
    len(planos_tratado)
)

print(
    "Combinações equipe × ciclo distintas em planos:",
    len(pares_planos)
)

print(
    "Combinações de planos sem correspondência em resultados:",
    len(pares_planos - pares_resultados)
)

print(
    "Combinações de planos sem correspondência no Eixo 3:",
    len(pares_planos - pares_eixo3)
)

print(
    "Combinações de planos sem correspondência no NPS:",
    len(pares_planos - pares_nps)
)

Registros em planos_acao: 420
Combinações equipe × ciclo distintas em planos: 420
Combinações de planos sem correspondência em resultados: 0
Combinações de planos sem correspondência no Eixo 3: 0
Combinações de planos sem correspondência no NPS: 0


#### Resultado da validação

- Os 420 registros de `planos_acao` correspondem a 420 combinações distintas de equipe e ciclo.
- Todas essas combinações possuem correspondência em `resultados_ciclos`, `eixo3_indicadores` e `nps_atributos`, não sendo identificados planos associados a combinações inexistentes nas demais bases.
- Dessa forma, a integridade referencial dos planos de ação foi confirmada.

### 6.4 Consistência do Eixo 3 entre as tabelas

- A tabela `resultados_ciclos` apresenta o valor consolidado do Eixo 3 na coluna `eixo3_total`, enquanto `eixo3_indicadores` contém os pontos obtidos em cada indicador.
- Para avaliar a consistência entre as bases, os pontos dos indicadores serão somados por equipe e ciclo e comparados ao valor consolidado registrado em `resultados_ciclos`.

In [91]:
eixo3_soma = (
    eixo3_tratado
    .groupby(["codigo_ine", "ciclo"], as_index=False)
    ["pontos_obtidos"]
    .sum()
    .rename(columns={
        "pontos_obtidos": "eixo3_soma_indicadores"
    })
)

eixo3_soma.head()

,codigo_ine,ciclo,eixo3_soma_indicadores
0,6000000000,2023.1,395.8
1,6000000000,2023.2,357.7
2,6000000000,2023.3,380.1
3,6000000000,2024.1,354.3
4,6000000000,2024.2,327.4


In [92]:
comparacao_eixo3 = resultados_tratado[
    ["codigo_ine", "ciclo", "eixo3_total"]
].merge(
    eixo3_soma,
    on=["codigo_ine", "ciclo"],
    how="left"
)

comparacao_eixo3["diferenca"] = (
    comparacao_eixo3["eixo3_total"]
    - comparacao_eixo3["eixo3_soma_indicadores"]
).abs()

print(
    "Comparações realizadas:",
    comparacao_eixo3["diferenca"].notna().sum()
)

print(
    "Eixo 3 consolidado ausente:",
    comparacao_eixo3["eixo3_total"].isna().sum()
)

print(
    "Soma dos indicadores ausente:",
    comparacao_eixo3["eixo3_soma_indicadores"].isna().sum()
)

print(
    "Maior diferença encontrada:",
    comparacao_eixo3["diferenca"].max()
)

comparacao_eixo3["diferenca"].describe()

Comparações realizadas: 1306
Eixo 3 consolidado ausente: 57
Soma dos indicadores ausente: 0
Maior diferença encontrada: 5.684341886080802e-14


count    1.306000e+03
mean     3.003213e-15
std      1.272075e-14
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.684342e-14
Name: diferenca, dtype: float64

In [93]:
tolerancia = 0.01

divergencias_eixo3 = comparacao_eixo3[
    comparacao_eixo3["diferenca"] > tolerancia
]

print(
    "Divergências superiores à tolerância:",
    len(divergencias_eixo3)
)

Divergências superiores à tolerância: 0


#### Resultado da validação

- Foram realizadas 1.306 comparações entre o valor consolidado do Eixo 3 e a soma dos pontos dos indicadores.
- A maior diferença observada foi de aproximadamente `5,68 × 10⁻¹⁴`, valor decorrente da precisão numérica de operações com números de ponto flutuante e sem relevância analítica. Considerando tolerância de 0,01 ponto, não foram identificadas divergências entre o valor consolidado e a soma dos indicadores.
- Foram observadas 57 combinações em que `eixo3_total` está ausente em `resultados_ciclos`, embora o detalhamento dos indicadores esteja disponível em `eixo3_indicadores`. Esses valores não foram preenchidos automaticamente, preservando-se a ausência registrada na tabela consolidada.

### 6.5 Validação final das tabelas preparadas

- Após os tratamentos e as verificações de consistência entre as tabelas, é realizada uma validação final para confirmar a estrutura das bases preparadas antes de sua disponibilização para as etapas posteriores de análise e visualização.
- Essa verificação resume aspectos relacionados a dimensões, unicidade das chaves, integridade referencial e cobertura das combinações entre equipe e ciclo.

In [94]:
pares_resultados = set(
    resultados_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

pares_eixo3 = set(
    eixo3_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

pares_nps = set(
    nps_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

pares_planos = set(
    planos_tratado[
        ["codigo_ine", "ciclo"]
    ].itertuples(index=False, name=None)
)

In [95]:
print("===== VALIDAÇÃO FINAL =====")

print("\n--- EQUIPES ---")
print("Dimensão:", equipes_tratado.shape)
print("INEs distintos:", equipes_tratado["codigo_ine"].nunique())
print(
    "INEs duplicados:",
    equipes_tratado["codigo_ine"].duplicated().sum()
)

print("\n--- RESULTADOS_CICLOS ---")
print("Dimensão:", resultados_tratado.shape)
print(
    "Chaves duplicadas:",
    resultados_tratado.duplicated(
        subset=["codigo_ine", "ciclo"]
    ).sum()
)
print(
    "INEs sem cadastro:",
    len(
        set(resultados_tratado["codigo_ine"])
        - set(equipes_tratado["codigo_ine"])
    )
)

print("\n--- EIXO3_INDICADORES ---")
print("Dimensão:", eixo3_tratado.shape)
print(
    "Chaves duplicadas:",
    eixo3_tratado.duplicated(
        subset=["codigo_ine", "ciclo", "codigo_indicador"]
    ).sum()
)
print(
    "INEs sem cadastro:",
    len(
        set(eixo3_tratado["codigo_ine"])
        - set(equipes_tratado["codigo_ine"])
    )
)

print("\n--- NPS_ATRIBUTOS ---")
print("Dimensão:", nps_tratado.shape)
print(
    "Chaves duplicadas:",
    nps_tratado.duplicated(
        subset=["codigo_ine", "ciclo", "codigo_atributo"]
    ).sum()
)
print(
    "INEs sem cadastro:",
    len(
        set(nps_tratado["codigo_ine"])
        - set(equipes_tratado["codigo_ine"])
    )
)

print("\n--- PLANOS_ACAO ---")
print("Dimensão:", planos_tratado.shape)
print(
    "IDs duplicados:",
    planos_tratado["id_plano"].duplicated().sum()
)
print(
    "INEs sem cadastro:",
    len(
        set(planos_tratado["codigo_ine"])
        - set(equipes_tratado["codigo_ine"])
    )
)

print("\n--- INTEGRAÇÃO ---")
print(
    "Combinações equipe × ciclo em resultados:",
    len(pares_resultados)
)
print(
    "Combinações equipe × ciclo em Eixo 3:",
    len(pares_eixo3)
)
print(
    "Combinações equipe × ciclo em NPS:",
    len(pares_nps)
)
print(
    "Planos sem correspondência em resultados:",
    len(pares_planos - pares_resultados)
)
print(
    "Divergências no Eixo 3:",
    len(divergencias_eixo3)
)

===== VALIDAÇÃO FINAL =====

--- EQUIPES ---
Dimensão: (132, 9)
INEs distintos: 132
INEs duplicados: 0

--- RESULTADOS_CICLOS ---
Dimensão: (1363, 13)
Chaves duplicadas: 0
INEs sem cadastro: 0

--- EIXO3_INDICADORES ---
Dimensão: (13294, 7)
Chaves duplicadas: 0
INEs sem cadastro: 0

--- NPS_ATRIBUTOS ---
Dimensão: (6815, 6)
Chaves duplicadas: 0
INEs sem cadastro: 0

--- PLANOS_ACAO ---
Dimensão: (420, 9)
IDs duplicados: 0
INEs sem cadastro: 0

--- INTEGRAÇÃO ---
Combinações equipe × ciclo em resultados: 1363
Combinações equipe × ciclo em Eixo 3: 1363
Combinações equipe × ciclo em NPS: 1363
Planos sem correspondência em resultados: 0
Divergências no Eixo 3: 0


### 6.6 Síntese da preparação dos dados

- Após a etapa de preparação, as cinco tabelas foram submetidas a tratamentos e validações individuais e integradas.
- Os principais tratamentos realizados incluíram a padronização de identificadores e categorias, normalização dos ciclos de monitoramento, conversão de tipos de dados, tratamento de duplicidades, correção de inconsistências determinísticas e preservação de valores ausentes quando não havia informação suficiente para sua reconstrução.
- Na validação integrada, foram identificados e removidos 13 registros órfãos de `resultados_ciclos`, cujos códigos de equipe não possuíam correspondência na tabela cadastral `equipes`. Após esse tratamento, `resultados_ciclos`, `eixo3_indicadores` e `nps_atributos` passaram a apresentar exatamente as mesmas 1.363 combinações de equipe e ciclo.
- Os 420 planos de ação possuem correspondência válida nas bases analíticas. Também foi confirmada a consistência entre o valor consolidado do Eixo 3 e a soma dos pontos de seus indicadores, sem divergências superiores à tolerância adotada.
- Os valores ausentes sem regra segura de reconstrução foram preservados, evitando imputações arbitrárias. Dessa forma, as bases preparadas mantêm rastreabilidade em relação aos dados originais e encontram-se estruturadas para as etapas posteriores de análise e visualização.

## 7. Geração das bases tratadas

- As etapas anteriores documentam o processo de preparação, limpeza e validação dos dados.
- A geração oficial das bases tratadas foi centralizada no pipeline disponível em `src/tratamento.py`, evitando duplicidade entre o notebook e o código-fonte.
- O pipeline pode ser executado a partir da raiz do projeto com:

`python src/tratamento.py`

- Os arquivos resultantes são armazenados na pasta `data/processed/`, mantendo os dados originais preservados na pasta `data/raw/`
- Dessa forma, este notebook documenta as decisões de tratamento, enquanto o módulo em `src/` é responsável pela execução reproduzível do processo.